## Caso de Uso: Arquitetura de Implantação Híbrida

### Cenário: Processamento Seguro de Dados com Acesso à Internet

Neste padrão de implantação, separamos os componentes do AgentCore para equilibrar segurança e funcionalidade:

**AgentCore Browser (Modo Público)**
- Implantado em sub-redes públicas com acesso à internet
- Permite navegação em sites e serviços publicamente acessíveis
- Gerencia integrações externas e recuperação de dados em tempo real
- Beneficia-se de conectividade direta à internet para operações em tempo real

**AgentCore Runtime (Modo VPC)**
- Implantado em sub-redes privadas dentro de uma VPC segura
- Processa dados sensíveis e lógica de negócio
- Mantém isolamento estrito de rede
- Comunica-se com o componente de navegador através de canais internos seguros

### Benefícios

- **Segurança**: O processamento sensível permanece isolado na rede privada
- **Performance**: Operações do navegador obtêm acesso direto à internet sem sobrecarga de NAT
- **Conformidade**: Atende requisitos regulatórios para isolamento de dados
- **Escalabilidade**: Cada componente pode escalar independentemente com base nas demandas de carga de trabalho

### Fluxo da Arquitetura

![image.png](Architecture-vpc-browser.png)

# Instruções de Execução da Stack CloudFormation

## Pré-requisitos
- AWS CLI configurada com permissões apropriadas
- Arquivo de template CloudFormation pronto

## Passos de Execução

### 1. Execute o passo abaixo para lançar o CFN (~13 Mins)

In [ ]:
import boto3
import yaml
import os

# Variáveis de configuração
region = 'us-east-1'  # Altere para a região desejada
template_path = 'cfn-vpc-browser.yaml'
stack_name = 'vpc-browser-stack-new'

# Inicializar cliente CloudFormation com região configurável
cf_client = boto3.client('cloudformation', region_name=region)

# Ler o template CloudFormation
with open(template_path, 'r') as template_file:
    template_body = template_file.read()

try:
    # Criar a stack CloudFormation
    response = cf_client.create_stack(
        StackName=stack_name,
        TemplateBody=template_body,
        Capabilities=['CAPABILITY_IAM', 'CAPABILITY_NAMED_IAM']
    )
    
    print(f"Criação da stack iniciada na região: {region}")
    print(f"Stack ID: {response['StackId']}")
    
    # Aguardar conclusão da criação da stack
    waiter = cf_client.get_waiter('stack_create_complete')
    print("Aguardando conclusão da criação da stack...")
    waiter.wait(StackName=stack_name)
    
    print(f"Stack '{stack_name}' criada com sucesso em {region}!")
    
except Exception as e:
    print(f"Erro ao criar stack: {str(e)}")

### 2. Instruções para Teste

In [ ]:
import boto3
import subprocess
import json
from IPython.display import display, Markdown

# Buscar ARN do AgentRuntime da saída do CloudFormation
def get_cfn_output(stack_name, output_key, region='us-east-1'):
    """Buscar valor de saída da stack CloudFormation"""
    cfn = boto3.client('cloudformation', region_name=region)
    try:
        response = cfn.describe_stacks(StackName=stack_name)
        outputs = response['Stacks'][0]['Outputs']
        for output in outputs:
            if output['OutputKey'] == output_key:
                return output['OutputValue']
    except Exception as e:
        print(f"Erro ao buscar saída do CFN: {e}")
    return None

# Obter o ARN do AgentRuntime (usando stack_name da célula anterior)
agent_runtime_arn = get_cfn_output(stack_name, 'AgentRuntimeArn')
agent_runtime_id = get_cfn_output(stack_name, 'AgentRuntimeId')
development_instance = get_cfn_output(stack_name, 'DevelopmentInstanceId')
web_server_ip = get_cfn_output(stack_name, 'WebServerPrivateIp')


# Gerar instruções completas de teste
instructions = f"""# Instruções de Teste para Bedrock Agent Runtime

## Pré-requisitos
- Stack CloudFormation foi concluída com sucesso

## Processo de Teste Passo a Passo

### 1. Conectar à Instância EC2
Conecte-se à instância EC2 `{development_instance}` via Browser Connector ou SSH

### 2. Configurar Ambiente
Execute os seguintes comandos na instância EC2:

```bash
sudo yum update -y

# Instalar ferramentas de desenvolvimento
sudo dnf install git -y && \\
curl -LsSf https://astral.sh/uv/install.sh | sh && \\
echo 'export PATH="$HOME/.cargo/bin:$PATH"' >> ~/.bashrc && \\
source ~/.bashrc && \\
echo 'Instalar Python Venv'

uv init vpc-browser --python 3.13 && cd vpc-browser
uv venv --python 3.13
source .venv/bin/activate
uv pip install boto3

cat > call-agent.py << 'EOF'
import boto3
import json

client = boto3.client('bedrock-agentcore', region_name='us-east-1')

payload = json.dumps({{
    "prompt": "Acesse {web_server_ip} via http na porta 8080 para verificar quais são os feriados em novembro"
}})

response = client.invoke_agent_runtime(
    agentRuntimeArn="{agent_runtime_arn}",
    runtimeSessionId='dfmeoagmreaklgmrkleafremoigrmtesogmtrskhmtkrlshmt',  # Deve ter 33+ caracteres
    payload=payload,
    qualifier="DEFAULT"  # Opcional
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Resposta do Agent:", response_data)
EOF


```

### 3. Executar o teste do agente
```
python call-agent.py
```

### 4. Monitorar logs no CloudWatch:
- Navegue até CloudWatch Logs no Console AWS
- Procure pelo grupo de logs: /aws/bedrock-agentcore/runtimes/{agent_runtime_id}
- Para visualizar navegação ao vivo, vá ao console -> Amazon Bedrock AgentCore -> Built-in Tools -> Browser tools -> browser_stack_browser -> View live session
- Monitore logs de execução em tempo real e quaisquer erros

"""

Markdown(instructions)

### 3. Limpeza

In [ ]:
import boto3

# Deletar a stack
cfn = boto3.client('cloudformation', region_name=region)
cfn.delete_stack(StackName=stack_name)

print(f"Deleção da stack '{stack_name}' iniciada na região '{region}'")

# Aguardar conclusão da deleção
waiter = cfn.get_waiter('stack_delete_complete')
print("Aguardando conclusão da deleção da stack...")
waiter.wait(StackName=stack_name)

print(f"Stack '{stack_name}' deletada com sucesso!")